In [10]:
!pip install psutil nvidia-ml-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.9 MB/s eta 0:00:00


In [1]:
import time
import psutil
import torch
import pandas as pd
from pathlib import Path

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
except Exception as e:
    print("NVML not available:", e)
    NVML_AVAILABLE = False
    GPU_HANDLE = None

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/MSC_PROJECT")
DATA_DIR = PROJECT_ROOT / "data" / "avdeepfake1mpp"
MANIFEST_DIR = DATA_DIR / "manifests"
RESULTS_DIR = DATA_DIR / "results"
CHECKPOINT_DIR = DATA_DIR / "checkpoints"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FULL_MANIFEST = MANIFEST_DIR / "usable_multimodal_val_labelled_manifest.csv"

df = pd.read_csv(FULL_MANIFEST)

print(df.shape)
print(df.columns.tolist())
display(df.head())

(76928, 25)
['path', 'relative_path', 'filename', 'file_size_bytes', 'file', 'original', 'split', 'modify_type', 'audio_model', 'fake_segments', 'audio_fake_segments', 'visual_fake_segments', 'video_frames', 'audio_frames', 'video_model', 'has_audio_stream', 'audio_decodable', 'audio_codec', 'sample_rate', 'channels', 'audio_duration', 'binary_label', 'visual_fake', 'audio_fake', 'condition']


,path,relative_path,filename,file_size_bytes,file,original,split,modify_type,audio_model,fake_segments,...,has_audio_stream,audio_decodable,audio_codec,sample_rate,channels,audio_duration,binary_label,visual_fake,audio_fake,condition
0,/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/e...,lrs3/1Hok1iGFNSk/00004/fake_video_fake_audio.mp4,fake_video_fake_audio.mp4,1524347,lrs3/1Hok1iGFNSk/00004/fake_video_fake_audio.mp4,lrs3/1Hok1iGFNSk/00004/real.mp4,val,both_modified,yourtts,"[[0.65, 1.15], [2.03, 2.25]]",...,True,True,pcm_s16le,16000.0,1.0,18.176000,1,1,1,fake_video_fake_audio
1,/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/e...,lrs3/1Hok1iGFNSk/00004/fake_video_real_audio.mp4,fake_video_real_audio.mp4,1517990,lrs3/1Hok1iGFNSk/00004/fake_video_real_audio.mp4,lrs3/1Hok1iGFNSk/00004/real.mp4,val,visual_modified,NaN,"[[0.65, 0.97], [1.851, 2.151]]",...,True,True,pcm_s16le,16000.0,1.0,18.048000,1,1,0,fake_video_real_audio
2,/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/e...,lrs3/1Hok1iGFNSk/00004/real.mp4,real.mp4,2223274,lrs3/1Hok1iGFNSk/00004/real.mp4,lrs3/1Hok1iGFNSk/00004/real.mp4,val,real,NaN,[],...,True,True,pcm_s16le,16000.0,1.0,17.920000,0,0,0,real
3,/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/e...,lrs3/1Hok1iGFNSk/00004/real_video_fake_audio.mp4,real_video_fake_audio.mp4,1527373,lrs3/1Hok1iGFNSk/00004/real_video_fake_audio.mp4,lrs3/1Hok1iGFNSk/00004/real.mp4,val,audio_modified,yourtts,"[[0.65, 1.15], [2.03, 2.25]]",...,True,True,pcm_s16le,16000.0,1.0,18.176000,1,0,1,real_video_fake_audio
4,/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/e...,lrs3/2YPHJdq5QWM/00001/00001_p1.mp4,00001_p1.mp4,2450165,lrs3/2YPHJdq5QWM/00001/00001_p1.mp4,NaN,val,real,NaN,[],...,True,True,pcm_s16le,16000.0,1.0,50.596313,0,0,0,real


In [3]:
PATH_COL = "path"
LABEL_COL = "binary_label"

print(df[LABEL_COL].value_counts(dropna=False))

binary_label
1    56907
0    20021
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

df = df[df[LABEL_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

MAX_SAMPLES = 20000

if len(df) > MAX_SAMPLES:
    df_used = df.groupby(LABEL_COL, group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), MAX_SAMPLES // df[LABEL_COL].nunique()),
            random_state=42
        )
    ).reset_index(drop=True)
else:
    df_used = df.reset_index(drop=True)

print("Using rows:", len(df_used))
print(df_used[LABEL_COL].value_counts())

train_df, temp_df = train_test_split(
    df_used,
    test_size=0.30,
    random_state=42,
    stratify=df_used[LABEL_COL]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df[LABEL_COL]
)

train_csv = MANIFEST_DIR / "train_visual_binary.csv"
val_csv = MANIFEST_DIR / "val_visual_binary.csv"
test_csv = MANIFEST_DIR / "test_visual_binary.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

/tmp/ipykernel_1777626/3915911300.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_used = df.groupby(LABEL_COL, group_keys=False).apply(


Using rows: 20000
binary_label
0    10000
1    10000
Name: count, dtype: int64
Train: 14000
Val: 3000
Test: 3000


In [5]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class VideoFrameDataset(Dataset):
    def __init__(self, csv_path, path_col, label_col, num_frames=16, image_size=224):
        self.df = pd.read_csv(csv_path)
        self.path_col = path_col
        self.label_col = label_col
        self.num_frames = num_frames

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def _sample_frames(self, video_path):
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            raise RuntimeError(f"No frames found: {video_path}")

        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()

            if not ret:
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = self.transform(frame)
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            raise RuntimeError(f"No frames sampled: {video_path}")

        while len(frames) < self.num_frames:
            frames.append(frames[-1])

        return torch.stack(frames)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        frames = self._sample_frames(row[self.path_col])
        label = int(row[self.label_col])

        return frames, torch.tensor(label, dtype=torch.long)

In [6]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

class FrameAverageConvNeXt(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        self.backbone = convnext_tiny(weights=weights)

        in_features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape

        x = x.reshape(B * T, C, H, W)

        frame_logits = self.backbone(x)          # [B*T, 2]
        frame_logits = frame_logits.reshape(B, T, -1)

        video_logits = frame_logits.mean(dim=1)  # [B, 2]

        return video_logits

In [7]:
from torch.utils.data import DataLoader

NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 16

train_dataset = VideoFrameDataset(
    train_csv,
    path_col=PATH_COL,
    label_col=LABEL_COL,
    num_frames=NUM_FRAMES,
    image_size=IMAGE_SIZE
)

val_dataset = VideoFrameDataset(
    val_csv,
    path_col=PATH_COL,
    label_col=LABEL_COL,
    num_frames=NUM_FRAMES,
    image_size=IMAGE_SIZE
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

In [8]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = FrameAverageConvNeXt(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

Device: cuda


In [9]:
from tqdm.auto import tqdm
import time

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    epoch=None,
    total_epochs=None,
    run_name=None,
    results_dir=None,
    hardware_log_rows=None,
    log_every=25,
):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    desc = f"Train {epoch}/{total_epochs}" if epoch is not None else "Train"

    progress_bar = tqdm(
        loader,
        desc=desc,
        total=len(loader),
        leave=True,
        dynamic_ncols=True,
        mininterval=1.0,
    )

    epoch_start_time = time.time()

    for batch_idx, (frames, labels) in enumerate(progress_bar):
        batch_start_time = time.time()

        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(frames)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_time = time.time() - batch_start_time

        total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        # Only update visible metrics every N batches, not every batch.
        if batch_idx % log_every == 0 or batch_idx == len(loader) - 1:
            running_loss = total_loss / max(1, len(all_labels))
            running_acc = accuracy_score(all_labels, all_preds)
            running_f1 = f1_score(all_labels, all_preds, zero_division=0)

            gpu_allocated = (
                torch.cuda.memory_allocated() / 1024**3
                if device == "cuda" else 0
            )

            progress_bar.set_postfix_str(
                f"loss={running_loss:.4f} | "
                f"acc={running_acc:.4f} | "
                f"f1={running_f1:.4f} | "
                f"gpu={gpu_allocated:.1f}GB | "
                f"batch={batch_time:.2f}s"
            )

        # Hardware logging can still happen every N batches.
        if hardware_log_rows is not None and batch_idx % log_every == 0:
            hw = get_hardware_snapshot(device=device)

            hw.update({
                "run_name": run_name,
                "phase": "train",
                "epoch": epoch,
                "batch_idx": batch_idx,
                "batch_time_sec": batch_time,
                "running_loss": total_loss / max(1, len(all_labels)),
            })

            hardware_log_rows.append(hw)

            if results_dir is not None and run_name is not None:
                pd.DataFrame(hardware_log_rows).to_csv(
                    Path(results_dir) / f"{run_name}_hardware_log.csv",
                    index=False
                )

    epoch_time = time.time() - epoch_start_time

    return {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(all_labels, all_preds),
        "balanced_acc": balanced_accuracy_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "epoch_time_sec": epoch_time,
    }


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    split_name="Val",
    return_outputs=False,
    run_name=None,
    results_dir=None,
    hardware_log_rows=None,
    epoch=None,
    log_every=25,
):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    progress_bar = tqdm(
        loader,
        desc=split_name,
        total=len(loader),
        leave=True,
        dynamic_ncols=True,
        mininterval=1.0,
    )

    eval_start_time = time.time()

    for batch_idx, (frames, labels) in enumerate(progress_bar):
        batch_start_time = time.time()

        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(frames)
            loss = criterion(logits, labels)

        batch_time = time.time() - batch_start_time

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

        if batch_idx % log_every == 0 or batch_idx == len(loader) - 1:
            running_loss = total_loss / max(1, len(all_labels))
            running_acc = accuracy_score(all_labels, all_preds)
            running_f1 = f1_score(all_labels, all_preds, zero_division=0)

            progress_bar.set_postfix_str(
                f"loss={running_loss:.4f} | "
                f"acc={running_acc:.4f} | "
                f"f1={running_f1:.4f} | "
                f"batch={batch_time:.2f}s"
            )

        if hardware_log_rows is not None and batch_idx % log_every == 0:
            hw = get_hardware_snapshot(device=device)

            hw.update({
                "run_name": run_name,
                "phase": split_name,
                "epoch": epoch,
                "batch_idx": batch_idx,
                "batch_time_sec": batch_time,
                "running_loss": total_loss / max(1, len(all_labels)),
            })

            hardware_log_rows.append(hw)

            if results_dir is not None and run_name is not None:
                pd.DataFrame(hardware_log_rows).to_csv(
                    Path(results_dir) / f"{run_name}_hardware_log.csv",
                    index=False
                )

    eval_time = time.time() - eval_start_time

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(all_labels, all_preds),
        "balanced_acc": balanced_accuracy_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "eval_time_sec": eval_time,
    }

    try:
        metrics["auc"] = roc_auc_score(all_labels, all_probs)
    except ValueError:
        metrics["auc"] = None

    if return_outputs:
        return metrics, all_labels, all_preds, all_probs

    return metrics

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
"""
torch_gpu_allocated_gb = memory PyTorch tensors are actively using
torch_gpu_reserved_gb  = memory PyTorch has reserved/cached
nvml_gpu_used_gb       = what nvidia-smi sees as total used GPU memory
"""

def get_hardware_snapshot(device="cuda"):
    snapshot = {
        "timestamp": time.time(),
        "cpu_percent": psutil.cpu_percent(interval=None),
        "ram_used_gb": psutil.virtual_memory().used / 1024**3,
        "ram_total_gb": psutil.virtual_memory().total / 1024**3,
        "ram_percent": psutil.virtual_memory().percent,
    }

    if device == "cuda" and torch.cuda.is_available():
        snapshot["torch_gpu_allocated_gb"] = torch.cuda.memory_allocated() / 1024**3
        snapshot["torch_gpu_reserved_gb"] = torch.cuda.memory_reserved() / 1024**3
        snapshot["torch_gpu_max_allocated_gb"] = torch.cuda.max_memory_allocated() / 1024**3
    else:
        snapshot["torch_gpu_allocated_gb"] = None
        snapshot["torch_gpu_reserved_gb"] = None
        snapshot["torch_gpu_max_allocated_gb"] = None

    if NVML_AVAILABLE:
        mem = pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(GPU_HANDLE)

        snapshot["nvml_gpu_used_gb"] = mem.used / 1024**3
        snapshot["nvml_gpu_total_gb"] = mem.total / 1024**3
        snapshot["nvml_gpu_util_percent"] = util.gpu
        snapshot["nvml_gpu_mem_util_percent"] = util.memory

        try:
            snapshot["gpu_temp_c"] = pynvml.nvmlDeviceGetTemperature(
                GPU_HANDLE,
                pynvml.NVML_TEMPERATURE_GPU
            )
        except Exception:
            snapshot["gpu_temp_c"] = None

        try:
            snapshot["gpu_power_w"] = pynvml.nvmlDeviceGetPowerUsage(GPU_HANDLE) / 1000
        except Exception:
            snapshot["gpu_power_w"] = None
    else:
        snapshot["nvml_gpu_used_gb"] = None
        snapshot["nvml_gpu_total_gb"] = None
        snapshot["nvml_gpu_util_percent"] = None
        snapshot["nvml_gpu_mem_util_percent"] = None
        snapshot["gpu_temp_c"] = None
        snapshot["gpu_power_w"] = None

    return snapshot

In [11]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def plot_training_history(history_df, save_dir, run_name):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Loss curve
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{run_name}: Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f"{run_name}_loss.png", dpi=200)
    plt.show()

    # Main performance curve
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_f1"], marker="o", label="Train F1")
    plt.plot(history_df["epoch"], history_df["val_f1"], marker="o", label="Val F1")
    plt.plot(history_df["epoch"], history_df["train_balanced_acc"], marker="o", label="Train balanced acc")
    plt.plot(history_df["epoch"], history_df["val_balanced_acc"], marker="o", label="Val balanced acc")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{run_name}: F1 and balanced accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f"{run_name}_f1_balanced_acc.png", dpi=200)
    plt.show()

    # Precision/recall curve
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["val_precision"], marker="o", label="Val precision")
    plt.plot(history_df["epoch"], history_df["val_recall"], marker="o", label="Val recall")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{run_name}: Validation precision and recall")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f"{run_name}_precision_recall.png", dpi=200)
    plt.show()

    # AUC curve, if available
    if "val_auc" in history_df.columns and history_df["val_auc"].notna().any():
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["val_auc"], marker="o", label="Val AUC")
        plt.xlabel("Epoch")
        plt.ylabel("AUC")
        plt.title(f"{run_name}: Validation AUC")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_dir / f"{run_name}_auc.png", dpi=200)
        plt.show()

In [ ]:
RUN_NAME = f"visual_convnext_tiny_{NUM_FRAMES}frames_balanced20k"

EPOCHS = 3
best_val_f1 = -1
history = []
hardware_log_rows = []

torch.cuda.reset_peak_memory_stats()

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")
    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch=epoch,
        total_epochs=EPOCHS,
        run_name=RUN_NAME,
        results_dir=RESULTS_DIR,
        hardware_log_rows=hardware_log_rows,
        log_every=10,
    )

    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        split_name=f"Validation epoch {epoch}/{EPOCHS}",
        run_name=RUN_NAME,
        results_dir=RESULTS_DIR,
        hardware_log_rows=hardware_log_rows,
        epoch=epoch,
        log_every=10,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / f"{RUN_NAME}_history.csv", index=False)

    pd.DataFrame(hardware_log_rows).to_csv(
        RESULTS_DIR / f"{RUN_NAME}_hardware_log.csv",
        index=False
    )

    print(row)

    plot_training_history(history_df, RESULTS_DIR, RUN_NAME)

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]

        checkpoint_path = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "num_frames": NUM_FRAMES,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "model": "frame_average_convnext_tiny",
            "run_name": RUN_NAME,
            "gpu_peak_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else None,
        }, checkpoint_path)

        print("Saved best checkpoint:", checkpoint_path)

hardware_df = pd.read_csv(RESULTS_DIR / f"{RUN_NAME}_hardware_log.csv")

display(hardware_df.head())
print(hardware_df.columns.tolist())


========== Epoch 1/3 ==========


Train 1/3:  14%|█▍        | 125/875 [06:48<29:45,  2.38s/it, loss=0.6227 | acc=0.6219 | f1=0.6168 | gpu=0.6GB | batch=0.24s] 

In [ ]:
import matplotlib.pyplot as plt

hardware_log_path = RESULTS_DIR / f"{RUN_NAME}_hardware_log.csv"
hardware_df = pd.read_csv(hardware_log_path)

display(hardware_df.head())
print(hardware_df.columns.tolist())

#GPU utilisation plot
plt.figure(figsize=(10, 5))
plt.plot(hardware_df["nvml_gpu_util_percent"])
plt.xlabel("Logged step")
plt.ylabel("GPU utilisation (%)")
plt.title(f"{RUN_NAME}: GPU utilisation")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{RUN_NAME}_gpu_utilisation.png", dpi=200)
plt.show()

#GPU memory plot
plt.figure(figsize=(10, 5))
plt.plot(hardware_df["torch_gpu_allocated_gb"], label="PyTorch allocated")
plt.plot(hardware_df["torch_gpu_reserved_gb"], label="PyTorch reserved")
plt.plot(hardware_df["nvml_gpu_used_gb"], label="NVML used")
plt.xlabel("Logged step")
plt.ylabel("GPU memory (GB)")
plt.title(f"{RUN_NAME}: GPU memory usage")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{RUN_NAME}_gpu_memory.png", dpi=200)
plt.show()

#CPU/RAM usage plot
plt.figure(figsize=(10, 5))
plt.plot(hardware_df["ram_percent"], label="RAM used (%)")
plt.plot(hardware_df["cpu_percent"], label="CPU usage (%)")
plt.xlabel("Logged step")
plt.ylabel("Usage (%)")
plt.title(f"{RUN_NAME}: CPU and RAM usage")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{RUN_NAME}_cpu_ram_usage.png", dpi=200)
plt.show()

In [ ]:
checkpoint_path = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

test_metrics, y_true, y_pred, y_prob = evaluate(
    model,
    test_loader,
    criterion,
    device,
    split_name="Test",
    return_outputs=True,
    run_name=RUN_NAME,
    results_dir=RESULTS_DIR,
    hardware_log_rows=hardware_log_rows,
    epoch="test",
    log_every=10,
)

print(test_metrics)

pd.DataFrame([test_metrics]).to_csv(
    RESULTS_DIR / f"{RUN_NAME}_test_metrics.csv",
    index=False
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 6))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=["Real", "Fake"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

plt.title(f"{RUN_NAME}: Test confusion matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{RUN_NAME}_confusion_matrix.png", dpi=200)
plt.show()

# ROC curve
fig, ax = plt.subplots(figsize=(6, 6))

RocCurveDisplay.from_predictions(
    y_true,
    y_prob,
    ax=ax
)

plt.title(f"{RUN_NAME}: Test ROC curve")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{RUN_NAME}_roc_curve.png", dpi=200)
plt.show()

In [ ]:
test_results_df = pd.read_csv(test_csv).reset_index(drop=True)

test_results_df["y_true"] = y_true
test_results_df["y_pred"] = y_pred
test_results_df["fake_probability"] = y_prob

test_results_df.to_csv(
    RESULTS_DIR / f"{RUN_NAME}_test_predictions.csv",
    index=False
)

condition_rows = []

for condition, group in test_results_df.groupby("condition"):
    condition_rows.append({
        "condition": condition,
        "n": len(group),
        "acc": accuracy_score(group["y_true"], group["y_pred"]),
        "balanced_acc": balanced_accuracy_score(group["y_true"], group["y_pred"]),
        "f1": f1_score(group["y_true"], group["y_pred"], zero_division=0),
        "precision": precision_score(group["y_true"], group["y_pred"], zero_division=0),
        "recall": recall_score(group["y_true"], group["y_pred"], zero_division=0),
        "mean_fake_probability": group["fake_probability"].mean(),
    })

condition_metrics_df = pd.DataFrame(condition_rows)
condition_metrics_df.to_csv(
    RESULTS_DIR / f"{RUN_NAME}_condition_metrics.csv",
    index=False
)

display(condition_metrics_df)